In [26]:
import pandas as pd
import numpy as np


In [27]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# directory containing experiment folders
base_path = Path("./data")

# helper function to read CSV files
def load_csv(filepath):
    return pd.read_csv(filepath, header=None).values

# helper function: recursively find a file inside a folder
def find_file(root: Path, filename: str):
    return next(root.rglob(filename), None)

# data structure: {experiment_group: {calibration_folder: {translation, quaternion}}}
experiment_groups = {}

# iterate over each subdirectory of ./data (each is an experiment group)
for exp_group in sorted(base_path.iterdir()):
    if not exp_group.is_dir():
        continue

    experiment_groups[exp_group.name] = {}

    # walk through experiment group to find Calibration folders
    for path in exp_group.rglob("*"):
        if path.is_dir() and path.name.endswith("Calibration"):

            # recursively search inside Calibration folder
            trans_file = find_file(path, "translation_vector.csv")
            quat_file  = find_file(path, "rotation_quaternion.csv")

            if trans_file is not None and quat_file is not None:
                experiment_groups[exp_group.name][path.name] = {
                    "translation": load_csv(trans_file),
                    "quaternion": load_csv(quat_file),
                }

# display summary
print("Loaded experiment groups:")
for group, experiments in experiment_groups.items():
    print(f"  {group}: {len(experiments)} Calibration folders")


Loaded experiment groups:
  experiment_1Hz: 44 Calibration folders


In [ ]:
# Process each experiment group separately
for group_name, experiments in experiment_groups.items():
    print(f"\n{'='*50}")
    print(f"Experiment Group: {group_name}")
    print(f"{'='*50}")
    
    if not experiments:
        print("No Calibration folders found")
        continue
    
    # list of translation and rotation data for this group
    trans_list = []
    rot_list = []
    
    # populate lists
    for exp_name, data in experiments.items():
        trans_list.append(data["translation"])
        rot_list.append(data["quaternion"])
    
    # compute mean and stddev for translations
    trans_mean = np.mean(trans_list, axis=0)
    rot_mean = np.mean(rot_list, axis=0)
    trans_stddev = np.std(trans_list, axis=0)
    rot_stddev = np.std(rot_list, axis=0)
    
    # compute magnitudes
    translation_distance = np.linalg.norm(trans_mean)
    rotation_degree = np.linalg.norm(rot_mean) * (180.0 / np.pi)
    
    # compute mean of absolute sequential differences
    trans_magnitudes = [np.linalg.norm(t) for t in trans_list]
    trans_seq_diff_mean = 0.0
    if len(trans_magnitudes) > 1:
        trans_seq_diffs = [abs(trans_magnitudes[i+1] - trans_magnitudes[i]) for i in range(len(trans_magnitudes)-1)]
        trans_seq_diff_mean = np.mean(trans_seq_diffs)
    
    # display results for this group
    print(f"Number of Calibrations: {len(experiments)}")
    print(f"Mean Translation Magnitude (m): {translation_distance:.4f}")
    print(f"Mean Rotation Magnitude (degrees): {rotation_degree:.4f}")
    print(f"Mean Sequential Translation Difference (m): {trans_seq_diff_mean:.4f}")


Experiment Group: experiment_1Hz
Number of Calibrations: 44
Mean Translation Magnitude (cm): 2.5055
Mean Rotation Magnitude (degrees): 56.8572
Mean Sequential Translation Difference (cm): 0.1595


In [29]:
# iterate over each experiment group
for group_name, group_data in experiment_groups.items():
    # collect translations and quaternions for this group
    trans_list = []
    rot_list = []
    
    for calib in group_data.values():
        trans_list.append(calib["translation"])
        rot_list.append(calib["quaternion"])
    
    # convert to numpy arrays
    trans_array = np.array(trans_list)
    rot_array = np.array(rot_list)
    
    # compute mean and stddev
    trans_mean = np.mean(trans_array, axis=0)
    rot_mean = np.mean(rot_array, axis=0)
    trans_stddev = np.std(trans_array, axis=0)
    rot_stddev = np.std(rot_array, axis=0)
    
    # display results for this experiment group
    print(f"Experiment Group: {group_name}")
    print("  Mean Translation:\n", trans_mean)
    print("  Mean Quaternion:\n", rot_mean)
    print("  Translation StdDev:\n", trans_stddev)
    print("  Quaternion StdDev:\n", rot_stddev)
    print("-" * 50)


Experiment Group: experiment_1Hz
  Mean Translation:
 [[-1.14845842 -2.22652091  0.03448689]]
  Mean Quaternion:
 [[0.01160697 0.00522983 0.17480909 0.97674463]]
  Translation StdDev:
 [[0.35639184 0.40866386 0.43801862]]
  Quaternion StdDev:
 [[0.06482933 0.06961576 0.07772763 0.01260348]]
--------------------------------------------------


In [ ]:
for group_name, group_data in experiment_groups.items():
    # collect translations and quaternions
    trans_list = []
    rot_list = []
    
    for calib in group_data.values():
        trans_list.append(calib["translation"])
        rot_list.append(calib["quaternion"])
    
    # convert to arrays
    trans_array = np.array(trans_list)
    rot_array = np.array(rot_list)
    
    # compute mean
    trans_mean = np.mean(trans_array, axis=0)
    rot_mean = np.mean(rot_array, axis=0)
    
    # compute distances/magnitudes
    translation_distance = np.linalg.norm(trans_mean)  # in whatever unit your CSV has
    rotation_degree = np.linalg.norm(rot_mean) * (180.0 / np.pi)  # radians → degrees
    
    # display
    print(f"Experiment Group: {group_name}")
    print("  Mean Translation Magnitude (m):", translation_distance)
    print("  Mean Rotation Magnitude (degrees):", rotation_degree)
    print("-" * 50)


Experiment Group: experiment_1Hz
  Mean Translation Magnitude (cm): 2.5055022366059574
  Mean Rotation Magnitude (degrees): 56.85723131264228
--------------------------------------------------


In [ ]:
for group_name, group_data in experiment_groups.items():
    # collect translations
    trans_list = [calib["translation"] for calib in group_data.values()]
    
    # convert to numpy arrays
    trans_array = np.array(trans_list)
    
    # mean translation
    trans_mean = np.mean(trans_array, axis=0)
    
    # translation magnitude
    translation_distance = np.linalg.norm(trans_mean)
    
    # compute sequential differences of magnitudes
    trans_magnitudes = [np.linalg.norm(t) for t in trans_list]
    
    if len(trans_magnitudes) > 1:
        trans_seq_diffs = [abs(trans_magnitudes[i+1] - trans_magnitudes[i])
                           for i in range(len(trans_magnitudes)-1)]
        trans_seq_diff_mean = np.mean(trans_seq_diffs)
    else:
        trans_seq_diff_mean = 0.0  # or np.nan if you prefer
    
    # display
    print(f"Experiment Group: {group_name}")
    print("  Mean Translation Magnitude (m):", translation_distance)
    print("  Mean of sequential absolute differences (m):", trans_seq_diff_mean)
    print("-" * 50)


Experiment Group: experiment_1Hz
  Mean Translation Magnitude (cm): 2.5055022366059574
  Mean of sequential absolute differences (cm): 0.1595235012705353
--------------------------------------------------
